# DJI Mini 4 Pro Fire Detection System

Jupyter notebook for forest fire detection using DJI Mini 4 Pro drone data. This notebook covers data acquisition, preprocessing, model training, detection, and visualization.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import logging

# DJI SDK and deep learning
try:
    from dji_sdk import Drone, Camera, FlightController
    print("✓ DJI SDK available")
except ImportError:
    print("⚠ DJI SDK not installed")

try:
    from ultralytics import YOLO
    print("✓ YOLOv8 available")
except ImportError:
    print("⚠ YOLOv8 not installed")

try:
    import tensorflow as tf
    print(f"✓ TensorFlow {tf.__version__} available")
except ImportError:
    print("⚠ TensorFlow not installed")

# Configure matplotlib for notebook display
%matplotlib inline
plt.style.use('dark_background')

# Set up logging
logging.basicConfig(level=logging.INFO)

In [ ]:
# For testing, we'll use a sample video file from data/raw folder
# In production, this would connect to the drone and stream live video

DATA_DIR = Path('../data/raw')
sample_video = list(DATA_DIR.glob('*.mp4')) or list(DATA_DIR.glob('*.avi'))

if sample_video:
    video_path = sample_video[0]
    print(f"Using sample video: {video_path}")
else:
    print("No sample videos found in data/raw/")
    print("Please add drone footage to data/raw/ folder")
    print("\nDummy video path for demonstration:")
    video_path = None  # None will trigger dataset creation below

# Alternative: Connect to real DJI Mini 4 Pro
# try:
#     drone = Drone()
#     camera = drone.camera
#     # Download flight log or stream video
#     video_stream = camera.start_recording()
# except Exception as e:
#     print(f"Could not connect to drone: {e}")

In [ ]:
def extract_frames(video_path, sample_rate=30, max_frames=100):
    """Extract frames from video file"""
    frames = []
    cap = cv2.VideoCapture(str(video_path))
    frame_count = 0
    
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_count % sample_rate == 0:
            frames.append(frame)
        frame_count += 1
    
    cap.release()
    print(f"Extracted {len(frames)} frames from video ({frame_count} total frames)")
    return frames

def preprocess_frame(frame, target_size=416):
    """Resize and normalize frame"""
    resized = cv2.resize(frame, (target_size, target_size))
    normalized = resized.astype(np.float32) / 255.0
    return normalized

def enhance_fire_colors(frame):
    """Enhance red/orange colors for fire detection"""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Red/orange color range
    lower_red1 = np.array([0, 100, 100])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([170, 100, 100])
    upper_red2 = np.array([180, 255, 255])
    
    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    fire_mask = cv2.bitwise_or(mask1, mask2)
    
    return fire_mask

# Test preprocessing
if video_path and Path(video_path).exists():
    frames = extract_frames(video_path, sample_rate=30, max_frames=10)
    
    if frames:
        # Show sample frame and preprocessing
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        sample_frame = frames[0]
        fire_mask = enhance_fire_colors(sample_frame)
        preprocessed = preprocess_frame(sample_frame)
        
        axes[0].imshow(cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original Frame')
        axes[0].axis('off')
        
        axes[1].imshow(fire_mask, cmap='hot')
        axes[1].set_title('Fire Color Enhancement')
        axes[1].axis('off')
        
        axes[2].imshow(preprocessed[:,:,0], cmap='gray')
        axes[2].set_title(f'Preprocessed ({preprocessed.shape})')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
else:
    print("No video file available for preprocessing demo")

In [ ]:
class FireDetectionModel:
    """Wrapper for fire detection model"""
    
    def __init__(self):
        self.model = None
        self.device = 'cpu'
    
    def load_pretrained(self, model_name='yolov8n.pt'):
        """Load pre-trained YOLOv8 model"""
        try:
            from ultralytics import YOLO
            self.model = YOLO(model_name)
            print(f"✓ Loaded {model_name}")
            return True
        except Exception as e:
            print(f"Could not load YOLOv8: {e}")
            return False
    
    def detect(self, frame, confidence=0.5):
        """Run detection on frame"""
        if self.model is None:
            return []
        
        results = self.model(frame)
        detections = []
        
        for r in results:
            if hasattr(r, 'boxes'):
                for box in r.boxes:
                    if float(box.conf[0]) >= confidence:
                        x1, y1, x2, y2 = map(int, box.xyxy[0])
                        conf = float(box.conf[0])
                        detections.append({
                            'bbox': (x1, y1, x2, y2),
                            'confidence': conf,
                            'class': 'fire'
                        })
        
        return detections

# Initialize model
detector = FireDetectionModel()

# Try to load model
model_loaded = detector.load_pretrained('yolov8n.pt')  # nano model for speed

if not model_loaded:
    print("Note: Install YOLOv8 with: pip install ultralytics")
    print("Or train custom model on fire dataset using PyTorch/TensorFlow")

In [ ]:
# Run detection on sample frames
if video_path and Path(video_path).exists() and detector.model:
    print("Running fire detection on sample frames...")
    frames = extract_frames(video_path, sample_rate=30, max_frames=5)
    
    all_detections = []
    for i, frame in enumerate(frames):
        detections = detector.detect(frame, confidence=0.3)
        all_detections.extend(detections)
        print(f"Frame {i}: Found {len(detections)} detections")
    
    print(f"\nTotal detections across frames: {len(all_detections)}")
    
elif video_path and Path(video_path).exists():
    print("Model not loaded; showing preprocessing only")
    frames = extract_frames(video_path, sample_rate=30, max_frames=5)
    
    # Demonstrate with color enhancement as pseudo-detection
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for i, frame in enumerate(frames[:6]):
        fire_mask = enhance_fire_colors(frame)
        # Simple thresholding to find fire regions
        _, thresh = cv2.threshold(fire_mask, 50, 255, cv2.THRESH_BINARY)
        
        # Find contours
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Draw on frame
        vis = frame.copy()
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area > 100:  # Filter small noise
                x, y, w, h = cv2.boundingRect(cnt)
                cv2.rectangle(vis, (x, y), (x+w, y+h), (0, 0, 255), 2)
        
        axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f'Frame {i}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("No video available for detection demo")

In [ ]:
def visualize_detections(frame, detections, fire_mask=None):
    """Draw detections on frame"""
    vis = frame.copy()
    
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        conf = det.get('confidence', 0)
        
        # Red box for fire
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
        
        # Label with confidence
        label = f"Fire {conf:.1%}"
        cv2.putText(vis, label, (x1, y1 - 5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    return vis

def create_heatmap(frame, detections):
    """Create heatmap from detection confidence"""
    heatmap = np.zeros(frame.shape[:2], dtype=np.float32)
    
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        conf = det.get('confidence', 0.5)
        heatmap[y1:y2, x1:x2] += conf
    
    # Normalize and apply colormap
    if heatmap.max() > 0:
        heatmap = (heatmap / heatmap.max() * 255).astype(np.uint8)
    
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    return heatmap_color

# Visualization example
if video_path and Path(video_path).exists():
    frames = extract_frames(video_path, sample_rate=30, max_frames=3)
    
    if frames:
        print("Creating visualization of results...")
        
        fig, axes = plt.subplots(len(frames), 2, figsize=(12, 4*len(frames)))
        if len(frames) == 1:
            axes = [axes]
        
        for i, frame in enumerate(frames):
            # Get fire color mask as simple detection
            fire_mask = enhance_fire_colors(frame)
            _, thresh = cv2.threshold(fire_mask, 50, 255, cv2.THRESH_BINARY)
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            # Create mock detections
            mock_detections = []
            for cnt in contours:
                area = cv2.contourArea(cnt)
                if area > 100:
                    x, y, w, h = cv2.boundingRect(cnt)
                    mock_detections.append({
                        'bbox': (x, y, x+w, y+h),
                        'confidence': min(1.0, area/1000)
                    })
            
            # Visualize
            vis_frame = visualize_detections(frame, mock_detections)
            heatmap = create_heatmap(frame, mock_detections)
            
            # Plot
            axes[i][0].imshow(cv2.cvtColor(vis_frame, cv2.COLOR_BGR2RGB))
            axes[i][0].set_title(f'Frame {i}: Detections ({len(mock_detections)} fires)')
            axes[i][0].axis('off')
            
            axes[i][1].imshow(cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB))
            axes[i][1].set_title(f'Frame {i}: Confidence Heatmap')
            axes[i][1].axis('off')
        
        plt.tight_layout()
        plt.show()
else:
    print("No video available for visualization demo")

## Next Steps

1. **Collect Training Data**: Gather labeled fire/no-fire images from your DJI Mini 4 Pro flights
2. **Fine-tune Model**: Train a custom YOLOv8 model on your specific fire data
3. **Optimize Performance**: Test different model sizes (nano/small/medium) for speed vs accuracy
4. **Real-time Integration**: Deploy detector to drone with `main.py` for live missions
5. **Geolocation**: Implement GPS-based fire point mapping
6. **Validation**: Test system in controlled environment before field deployment

### Resources

- [YOLOv8 Docs](https://docs.ultralytics.com/)
- [DJI Mini 4 Pro SDK](https://developer.dji.com/)
- [Fire Detection Datasets](https://github.com/topics/fire-detection)
- [OpenCV Tutorials](https://docs.opencv.org/)

## 6. Visualize Detection Results

Display detections with bounding boxes and heatmaps.

## 5. Detect Fires in Drone Footage

Apply detection model to identify fire regions in real-time.

## 4. Train or Load Fire Detection Model

Use a pre-trained YOLOv8 model or train custom model on fire dataset.

## 3. Preprocess Drone Footage

Extract frames from video, normalize, and prepare for detection.

## 2. Acquire Data from DJI Mini 4 Pro

Connect to the DJI Mini 4 Pro and download or stream video footage for analysis.

## 1. Import Required Libraries

Import necessary libraries for drone integration, image processing, and deep learning.